# NN Grid Research

## Setup

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

%load_ext autoreload
%autoreload 2

import time

## LUT Upload

In [ ]:
from src.lut.lut import LUT

LUT_PATH = "models/luts/W_all/lut_W_all.nc"

my_lut = LUT(LUT_PATH)
my_lut.load_data()

## Grid Search Algorithm
This section trains several models and saves it into a list.
The cell execute every combination.

This section takes several minutes to be computed.

In [ ]:
from src.nn.nn_model_from_lut import NN_model_from_LUT

NUM_LAYERS_TO_TRY = [2, 3, 4]
NUM_UNITS_TO_TRY = [8, 16, 32]
ACTIVATION_MODE = ["relu"]
OUTPUT_ACTIVATION_MODE = ["linear"]

EPOCHS = 500
BATCH_SIZE = 512
LEARNING_RATE = 3e-3#1e-3
MIN_DELTA = 0

PATH_GS = "models/nn/W_all/grid_search"

trained_models = []
W_to_visualize = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

print(f"Grid Search: {len(NUM_LAYERS_TO_TRY) * len(NUM_UNITS_TO_TRY) * len(ACTIVATION_MODE)} combinations...")
start_total = time.time()

for n_layers in NUM_LAYERS_TO_TRY:
    for n_units in NUM_UNITS_TO_TRY:
        for act_mode in ACTIVATION_MODE:
            for out_mode in OUTPUT_ACTIVATION_MODE:
                
                print("\n" + "="*60)
                print(f"TRAINING: {n_layers} Layer, {n_units} Neurons, {act_mode} Mode")
                print("="*60)

                model_name = f"{n_layers}L_{n_units}U_{act_mode}_{out_mode}_cosdecay"
                start_train = time.time()

                # --- directories ---
                path_plot = f"{PATH_GS}/{model_name}"
                path_export = f"{PATH_GS}/{model_name}"

                # --- NN SETUP ---
                current_model = NN_model_from_LUT(model_name, my_lut)
                current_model.define_structure(num_hidden_layers=n_layers,
                                               hidden_units=n_units,
                                               activation=act_mode,
                                               output_activation=out_mode,
                                               save_path_plots=path_plot)

                # --- NN TRAINING ---
                current_model.train(epochs=EPOCHS,
                                    batch_size=BATCH_SIZE,
                                    lr=LEARNING_RATE,
                                    min_delta=MIN_DELTA,
                                    save_path_plots=path_plot)
                train_time = time.time() - start_train
                print(f"Completed in {train_time:.1f} s.")

                # --- EXPORTS ---
                current_model.export(path_export)

                # --- Save errors_heatmap ---
                for W_k in W_to_visualize:
                    current_model.plot_error_heatmaps(considered_W = W_k,
                                                      show=False,
                                                      save_path=f"{path_plot}/heatmaps_error")


                # --- Append in the list ---
                #trained_models.append(current_model)


total_time = time.time() - start_total
print("\n" + "★"*60)
print(f"GRID SEARCH COMPLETED IN {total_time/60:.1f} MINUTES!")
print("★"*60)